# Phase 3 pilot: Qwen2.5-0.5B on free Colab

**Before running:** `Runtime → Change runtime type → T4 GPU`, then `Runtime → Run all`.

The repo is private, so cloning needs a GitHub token: click the 🔑 **Secrets** icon in the left sidebar, add a secret named `GH_TOKEN`
(a fine-grained token with read-only *Contents* access to `GTRAN7/vnpronouns`), and turn on notebook access.
If the repo is public, skip this.

Results are saved to Google Drive at `MyDrive/vnpronouns/outputs/Qwen2.5-0.5B-Instruct/`.
The last cell prints a summary to paste back to Claude.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import os, subprocess
try:
    from google.colab import userdata
    token = userdata.get('GH_TOKEN')
except Exception:
    token = None
url = f"https://{token}@github.com/GTRAN7/vnpronouns.git" if token else "https://github.com/GTRAN7/vnpronouns.git"
if not os.path.exists('/content/vnpronouns'):
    subprocess.run(['git', 'clone', '-q', url, '/content/vnpronouns'], check=True)
else:
    subprocess.run(['git', '-C', '/content/vnpronouns', 'pull', '-q'], check=True)
%cd /content/vnpronouns
!git log --oneline -1
!pip install -q -e .
!nvidia-smi --query-gpu=name,memory.total --format=csv

## 1. Baseline filter (60-question pool) + flip rates on the first 20 kept questions

In [ ]:
!python -m vnpronouns.run_behavior --config configs/colab_pilot.yaml --pool 60 --max-questions 20 2>&1 | tee /content/log_behavior.txt

## 2. Cache activations and build the senior − junior vector

In [ ]:
!python -m vnpronouns.extract --config configs/colab_pilot.yaml 2>&1 | tee /content/log_extract.txt

## 3. One steering run at the middle layer (12 of 24), plus a random-vector control

In [ ]:
!python -m vnpronouns.steer --config configs/colab_pilot.yaml --layer 12 --lams -2 -1 0 1 2 --vector deference 2>&1 | tee /content/log_steer.txt
!python -m vnpronouns.steer --config configs/colab_pilot.yaml --layer 12 --lams -2 -1 0 1 2 --vector random 2>&1 | tee -a /content/log_steer.txt

## 4. Summary to paste back to Claude

In [ ]:
import re
def clean(path):
    keep = []
    for line in open(path, encoding='utf-8'):
        if re.search(r'it/s\]|s/it\]|Warning|warn\(|^\s*$|Loading|Fetching|Downloading|%\|', line):
            continue
        keep.append(line.rstrip())
    return '\n'.join(keep)
for p in ['/content/log_behavior.txt', '/content/log_extract.txt', '/content/log_steer.txt']:
    print(f'===== {p.split("/")[-1]} =====')
    print(clean(p))
!ls -la /content/drive/MyDrive/vnpronouns/outputs/Qwen2.5-0.5B-Instruct/